In [ ]:
from pathlib import Path
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

DATA_DIR = Path(r"E:/marcel.furs/Documents/Data_Run_Walk/Data_Run_Walk/CLEAN_ANGLES_DATASET_FULLBODY")


LATENT_DIM = 16
WIDTH_FIRST = 256
BETA_KL = 0.01
BATCH_SIZE = 32
DROPOUT = 0.2
LEARNING_RATE = 1e-3
EPOCHS = 1000
PATIENCE = 20        

ANGLE_CHANNELS = [
    "LFootProgressAngles", "RFootProgressAngles",
    "LHipAngles",          "RHipAngles",
    "LKneeAngles",         "RKneeAngles",
    "LAnkleAngles",        "RAnkleAngles",
    "LForeFootAngles",     "RForeFootAngles",
    "LSpineAngles",        "RSpineAngles",
    "LShoulderAngles",     "RShoulderAngles",
    "LElbowAngles",        "RElbowAngles",
    "LWristAngles",        "RWristAngles",
    "LNeckAngles",         "RNeckAngles",
    "LPelvisAngles",       "RPelvisAngles",
    "LThoraxAngles",       "RThoraxAngles",
    "LHeadAngles",         "RHeadAngles",
]
NUM_JOINTS = len(ANGLE_CHANNELS)   # 26


X_train_M = np.load(DATA_DIR / "train_M_S1_fullbody.npy")      # (109, 100, 78)
X_test_M  = np.load(DATA_DIR / "test_M_S2_fullbody.npy")       # (105, 100, 78)

X_train_F = np.load(DATA_DIR / "train_F_S1_fullbody.npy")      # (124, 100, 78)
X_test_F  = np.load(DATA_DIR / "test_F_S2_fullbody.npy")       # (124, 100, 78)

# Subjects
subjects_train_M = np.load(DATA_DIR / "subjects_train_M_fullbody.npy", allow_pickle=True)
subjects_test_M  = np.load(DATA_DIR / "subjects_test_M_fullbody.npy",  allow_pickle=True)
subjects_train_F = np.load(DATA_DIR / "subjects_train_F_fullbody.npy", allow_pickle=True)
subjects_test_F  = np.load(DATA_DIR / "subjects_test_F_fullbody.npy",  allow_pickle=True)

print(f"  Mężczyźni Train (S1): {X_train_M.shape} - {len(np.unique(subjects_train_M))} osób")
print(f"  Mężczyźni Test (S2):  {X_test_M.shape}  - {len(np.unique(subjects_test_M))} osób")
print(f"  Kobiety Train (S1):   {X_train_F.shape} - {len(np.unique(subjects_train_F))} osób")
print(f"  Kobiety Test (S2):    {X_test_F.shape}  - {len(np.unique(subjects_test_F))} osób")

# =============================================================================
# 2️ FILTRUJ DA013 - tylko mężczyźni którzy mają OBA zbiory
# =============================================================================
print("\n" + "="*80)
print(" FILTROWANIE DA013 (brak w train_M_S1)")
print("="*80)

train_M_subjects = np.unique(subjects_train_M)
print(f"Mężczyźni w TRAIN: {len(train_M_subjects)} osób")
print(f"Lista: {sorted(train_M_subjects)}")

mask_test_M = np.isin(subjects_test_M, train_M_subjects)
X_test_M_filtered = X_test_M[mask_test_M]
subjects_test_M_filtered = subjects_test_M[mask_test_M]

removed = (~mask_test_M).sum()
print(f"\n Usunięto {removed} cykli z test_M (DA013)")
print(f"   Test M przed: {X_test_M.shape[0]} cykli")
print(f"   Test M po:    {X_test_M_filtered.shape[0]} cykli "
      f"({len(np.unique(subjects_test_M_filtered))} osób)")

# =============================================================================
# 3 POŁĄCZ WSZYSTKIE KOBIETY (S1 + S2) → ANOMALIE
# =============================================================================
print("\n" + "="*80)
print(" ŁĄCZENIE WSZYSTKICH KOBIET (anomalie)")
print("="*80)

X_female_all = np.concatenate([X_train_F, X_test_F], axis=0)
subjects_female_all = np.concatenate([subjects_train_F, subjects_test_F])

print(f"Wszystkie kobiety: {X_female_all.shape} - {len(np.unique(subjects_female_all))} osób")

# =============================================================================
# 4 PODZIAŁ TRAIN/VAL/TEST (subject-wise)
# =============================================================================
print("\n" + "="*80)
print(" PODZIAŁ TRAIN/VAL/TEST")
print("="*80)

unique_subjects_train_M = np.unique(subjects_train_M)
n_subjects_train = len(unique_subjects_train_M)
n_val_subjects = max(1, int(n_subjects_train * 0.2))

np.random.seed(42)
val_subjects_M = np.random.choice(unique_subjects_train_M, size=n_val_subjects, replace=False)
train_subjects_M_final = np.array([s for s in unique_subjects_train_M if s not in val_subjects_M])

mask_train_M = np.isin(subjects_train_M, train_subjects_M_final)
mask_val_M   = np.isin(subjects_train_M, val_subjects_M)

X_train_final = X_train_M[mask_train_M]
X_val         = X_train_M[mask_val_M]

# TEST Normal: Mężczyźni S2 (po filtracji)
X_test_normal  = X_test_M_filtered

# TEST Anomaly: Wszystkie kobiety (S1 + S2)
X_test_anomaly = X_female_all

print(f"\n Podział finalny (FULL BODY):")
print(f"  TRAIN (Mężczyźni S1 - 80%):  {X_train_final.shape[0]} cykli ({len(train_subjects_M_final)} osób)")
print(f"  VAL   (Mężczyźni S1 - 20%):  {X_val.shape[0]} cykli ({len(val_subjects_M)} osób)")
print(f"  TEST Normal (Mężczyźni S2):  {X_test_normal.shape[0]} cykli "
      f"({len(np.unique(subjects_test_M_filtered))} osób)")
print(f"  TEST Anomaly (Kobiety S1+S2): {X_test_anomaly.shape[0]} cykli "
      f"({len(np.unique(subjects_female_all))} osób)")

# =============================================================================
# 5️ PREPROCESSING (Flatten + Z-score)
# =============================================================================
print("\n" + "="*80)
print("🔧 PREPROCESSING (Flatten + Z-score)")
print("="*80)

# Flatten: (N, 100, 78) -> (N, 7800)
X_train_flat        = X_train_final.reshape(X_train_final.shape[0], -1)
X_val_flat          = X_val.reshape(X_val.shape[0], -1)
X_test_normal_flat  = X_test_normal.reshape(X_test_normal.shape[0], -1)
X_test_anomaly_flat = X_test_anomaly.reshape(X_test_anomaly.shape[0], -1)

print(f"Po flatten:")
print(f"  Train:        {X_train_flat.shape}")
print(f"  Val:          {X_val_flat.shape}")
print(f"  Test Normal:  {X_test_normal_flat.shape}")
print(f"  Test Anomaly: {X_test_anomaly_flat.shape}")

# Z-score fit tylko na TRAIN (Mężczyźni S1)
scaler = StandardScaler()
X_train_norm        = scaler.fit_transform(X_train_flat)
X_val_norm          = scaler.transform(X_val_flat)
X_test_normal_norm  = scaler.transform(X_test_normal_flat)
X_test_anomaly_norm = scaler.transform(X_test_anomaly_flat)

print(f"\n Po normalizacji:")
print(f"  Train:        mean={X_train_norm.mean():.3f}, std={X_train_norm.std():.3f}")
print(f"  Val:          mean={X_val_norm.mean():.3f}, std={X_val_norm.std():.3f}")
print(f"  Test Normal:  mean={X_test_normal_norm.mean():.3f}, std={X_test_normal_norm.std():.3f}")
print(f"  Test Anomaly: mean={X_test_anomaly_norm.mean():.3f}, std={X_test_anomaly_norm.std():.3f}")

input_dim = X_train_norm.shape[1]
print(f"\n Input dimension (FULL BODY): {input_dim}  # 100 × 78 = 7800")

# =============================================================================
# 6️ WIZUALIZACJA PRZYKŁADOWEGO STAWU (np. RAnkleAngles X)
# =============================================================================
print("\n" + "="*80)
print(" Wizualizacja przykładowego stawu (RAnkleAngles X, FULL BODY)")
print("="*80)

# Reshape do (N, 100, 26, 3)
X_train_vis        = X_train_final.reshape(X_train_final.shape[0], 100, NUM_JOINTS, 3)
X_test_normal_vis  = X_test_normal.reshape(X_test_normal.shape[0], 100, NUM_JOINTS, 3)
X_test_anomaly_vis = X_test_anomaly.reshape(X_test_anomaly.shape[0], 100, NUM_JOINTS, 3)

# znajdź indeks RAnkleAngles po nazwie (żeby było odpornie na zmiany listy)
joint_idx = ANGLE_CHANNELS.index("RAnkleAngles")
axis_idx  = 0   # X

plt.figure(figsize=(10, 6))
plt.plot(np.mean(X_train_vis[:, :, joint_idx, axis_idx], axis=0),
         label='Train (Mężczyźni S1)', linewidth=2.5, color='#3498db')
plt.plot(np.mean(X_test_normal_vis[:, :, joint_idx, axis_idx], axis=0),
         label='Test Normal (Mężczyźni S2)', linewidth=2.5, color='#2ecc71')
plt.plot(np.mean(X_test_anomaly_vis[:, :, joint_idx, axis_idx], axis=0),
         label='Test Anomaly (Kobiety S1+S2)', linewidth=2.5, color='#e74c3c')
plt.xlabel('Cykl [%]')
plt.ylabel('Kąt [°]')
plt.title('RAnkleAngles (oś X) - średnie dla każdego zbioru (FULL BODY)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# SETUP

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import backend as K

In [ ]:
# Create a sampling layer

In [ ]:
class Sampling(layers.Layer):
    """Reparameterization trick: z = μ + ε·σ,  ε∼N(0,I)."""
    
    def call(self, inputs):
        z_mean, z_logvar = inputs
        epsilon = tf.random.normal(tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_logvar) * epsilon

In [ ]:
# Build the encoder (

In [ ]:
encoder_inputs = keras.Input(shape=(input_dim,))
x = layers.Dense(WIDTH_FIRST, activation="relu")(encoder_inputs)
x = layers.Dropout(DROPOUT)(x)
x = layers.Dense(WIDTH_FIRST // 2, activation="relu")(x)
x = layers.Dropout(DROPOUT)(x)
z_mean = layers.Dense(LATENT_DIM, name="z_mean")(x)
z_log_var = layers.Dense(LATENT_DIM, name="z_log_var")(x)
z = Sampling()([z_mean, z_log_var])
encoder = keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")

print("\nENCODER:")
encoder.summary()

In [ ]:
# Build the decoder

In [ ]:
latent_input = keras.Input(shape=(LATENT_DIM,))
x = layers.Dense(WIDTH_FIRST // 2, activation="relu")(latent_input)
x = layers.Dropout(DROPOUT)(x)
x = layers.Dense(WIDTH_FIRST, activation="relu")(x)
x = layers.Dropout(DROPOUT)(x)
decoder_output = layers.Dense(input_dim, activation="linear")(x)
decoder = keras.Model(latent_input, decoder_output, name="decoder")

print("\nDecoder:")
decoder.summary()

In [ ]:
# Define the VAE as a Model with a custom train_step

In [ ]:
"""β-VAE: loss = MSE + β·KL."""
class VAE(keras.Model):
    def __init__(self, encoder, decoder, beta=BETA_KL, **kw):
        super().__init__(**kw)
        self.encoder     = encoder
        self.decoder     = decoder
        self.beta        = beta
        self.loss_tracker= keras.metrics.Mean(name="loss")
        self.mse_tracker = keras.metrics.Mean(name="mse")
        self.kl_tracker  = keras.metrics.Mean(name="kl")

    @property
    def metrics(self):          # <– Keras będzie to resetował co epokę
        return [self.loss_tracker, self.mse_tracker, self.kl_tracker]

    def call(self, x):
        _, _, z = self.encoder(x)
        return self.decoder(z)

    def train_step(self, x):
        with tf.GradientTape() as tape:
            mu, logvar, z = self.encoder(x, training=True)
            logvar = tf.clip_by_value(logvar, -10.0, 10.0)       #BEZ TEGO
            recon = self.decoder(z, training=True)
            mse = tf.reduce_mean(tf.square(x - recon), axis=1)
            kl  = -0.5 * tf.reduce_sum(1 + logvar - tf.square(mu) - tf.exp(logvar), axis=1)
            #loss = tf.reduce_mean(mse + self.beta * kl)
            loss = tf.reduce_mean(mse + self.beta * kl)

        grads = tape.gradient(loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.loss_tracker.update_state(loss)
        self.mse_tracker.update_state(tf.reduce_mean(mse))
        self.kl_tracker.update_state(tf.reduce_mean(kl))

        return {
            "loss": self.loss_tracker.result(),
            "mse":  self.mse_tracker.result(),
            "kl":   self.kl_tracker.result()
        }

    # val step (żeby działał val_loss)
    def test_step(self, x):
        mu, logvar, z = self.encoder(x, training=False)
        logvar = tf.clip_by_value(logvar, -10.0, 10.0)
        recon  = self.decoder(z, training=False)
        mse    = tf.reduce_mean(tf.square(x - recon), axis=1)
        kl     = -0.5 * tf.reduce_sum(1 + logvar - tf.square(mu) - tf.exp(logvar), axis=1)
        loss   = tf.reduce_mean(mse + self.beta * kl)
        self.loss_tracker.update_state(loss)
        self.mse_tracker.update_state(tf.reduce_mean(mse))
        self.kl_tracker.update_state(tf.reduce_mean(kl))
        return {
            "loss": self.loss_tracker.result(),
            "mse":  self.mse_tracker.result(),
            "kl":   self.kl_tracker.result()
        }




In [ ]:
# Train the VAE

In [ ]:
vae = VAE(encoder, decoder, beta=BETA_KL)
vae.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE, clipnorm=1.0))

# 6. train
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=PATIENCE,
    restore_best_weights=True,
    verbose=1
)

history = vae.fit(
    X_train_norm,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_data=(X_val_norm, None),
    callbacks=[early_stop],
    verbose=1
)



def compute_mse(model, X):
    mu, logvar, z = model.encoder.predict(X, verbose=0)
    X_recon = model.decoder.predict(z, verbose=0)
    return np.mean((X - X_recon)**2, axis=1)


mse_train = compute_mse(vae, X_train_norm)                   # Train (M S1 80%)
mse_val = compute_mse(vae, X_val_norm)                       # Val (M S1 20%)
mse_test_normal = compute_mse(vae, X_test_normal_norm)       # Test Normal 
mse_test_anomaly = compute_mse(vae, X_test_anomaly_norm)     # Test Anomaly 

print(f"  Train (M S1 80%):     {mse_train.mean():.6f} ± {mse_train.std():.6f}")
print(f"  Val (M S1 20%):       {mse_val.mean():.6f} ± {mse_val.std():.6f}")
print(f"  Test Normal (M S2):   {mse_test_normal.mean():.6f} ± {mse_test_normal.std():.6f}")
print(f"  Test Anomaly (F all): {mse_test_anomaly.mean():.6f} ± {mse_test_anomaly.std():.6f}")


y_train_val = np.concatenate([
    np.zeros(len(mse_train)),  # train = 0
    np.ones(len(mse_val))      # val = 1
])
mse_train_val = np.concatenate([mse_train, mse_val])

print(f"\nCombined train+val: {len(mse_train_val)} samples")

from sklearn.metrics import roc_curve, confusion_matrix

fpr_tv, tpr_tv, thresholds_tv = roc_curve(y_train_val, mse_train_val)


def f1_at_threshold(thr, y_true, y_score):
    y_pred = (y_score > thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    prec = tp / (tp + fp + 1e-9)
    rec = tp / (tp + fn + 1e-9)
    f1 = 2 * prec * rec / (prec + rec + 1e-9)
    
    return f1, prec, rec, tn, fp, fn, tp



from sklearn.metrics import roc_auc_score, average_precision_score

auc = roc_auc_score(y_true, y_score)
ap = average_precision_score(y_true, y_score)
separation = mse_test_anomaly.mean() / mse_test_normal.mean()

# Confusion matrix
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

# F1
test_prec = tp / (tp + fp + 1e-9)
test_rec = tp / (tp + fn + 1e-9)
test_f1 = 2 * test_prec * test_rec / (test_prec + test_rec + 1e-9)

# ROC dla wykresu
fpr, tpr, _ = roc_curve(y_true, y_score)


print(f"  AUC: {auc:.3f}")
print(f"  AP (Average Precision): {ap:.3f}")
print(f"  Separation (F/M): {separation:.3f}x")
print(f"  Threshold used: {threshold:.6f} (from train/val split)")

print(f"  True Negatives (M→M):  {tn:3d}/{len(mse_test_normal):3d} ({tn/len(mse_test_normal)*100:5.1f}%)")
print(f"  False Positives (M→F): {fp:3d}/{len(mse_test_normal):3d} ({fp/len(mse_test_normal)*100:5.1f}%)")
print(f"  False Negatives (F→M): {fn:3d}/{len(mse_test_anomaly):3d} ({fn/len(mse_test_anomaly)*100:5.1f}%)")
print(f"  True Positives (F→F):  {tp:3d}/{len(mse_test_anomaly):3d} ({tp/len(mse_test_anomaly)*100:5.1f}%)")

print(f"  TPR (Sensitivity/Recall): {test_rec:.3f}")
print(f"  TNR (Specificity):        {tn/(tn+fp):.3f}")
print(f"  FPR:                      {fp/(fp+tn):.3f}")
print(f"  FNR:                      {fn/(fn+tp):.3f}")
print(f"  Precision:                {test_prec:.3f}")
print(f"  F1-Score:                 {test_f1:.3f}")

# History
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history.history["mse"], label="Train MSE", linewidth=2, color='blue')
ax.plot(history.history["val_mse"], label="Val MSE", linewidth=2, color='cyan')
ax.plot(history.history["kl"], label="Train KL", linewidth=2, linestyle='--', color='orange')
ax.plot(history.history["val_kl"], label="Val KL", linewidth=2, linestyle='--', color='red')
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Loss", fontsize=12)
ax.set_title("Training History", fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Histogram
plt.figure(figsize=(10, 5))

plt.hist(mse_test_normal, bins=40, alpha=0.7, 
         label=f'Normal (M S2, n={len(mse_test_normal)})', 
         color='blue', edgecolor='black')
plt.hist(mse_test_anomaly, bins=40, alpha=0.7, 
         label=f'Anomaly (F all, n={len(mse_test_anomaly)})', 
         color='red', edgecolor='black')
plt.axvline(threshold, color='black', linestyle='--', linewidth=2, 
            label=f'Threshold')

plt.xlabel('Reconstruction Error (MSE)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Distribution of Reconstruction Errors (TEST SET)', 
          fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

textstr = f'AUC = {auc:.3f}\nAP = {ap:.3f}\nSep = {separation:.2f}x'
plt.text(0.98, 0.97, textstr, transform=plt.gca().transAxes,
         fontsize=11, ha='right', va='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

# ROC
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, linewidth=2, color='blue', label=f'Dense VAE (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow([[tn, fp], [fn, tp]], interpolation='nearest', cmap='Blues')
ax.figure.colorbar(im, ax=ax)
ax.set(xticks=[0, 1], yticks=[0, 1],
       xticklabels=['Male', 'Female'], yticklabels=['Male', 'Female'],
       ylabel='True label', xlabel='Predicted label',
       title=f'Confusion Matrix  thr={threshold:.4f}')
for i in range(2):
    for j in range(2):
        val = [[tn, fp], [fn, tp]][i][j]
        ax.text(j, i, val, ha="center", va="center",
                color="white" if val > (tn+tp)/2 else "black",
                fontsize=20)
plt.tight_layout()
plt.show()

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 8))
ax.axis('tight')
ax.axis('off')